In [1]:
pip install pmdarima

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 11.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 89.1 MB/s  0:00:00
  Attempting uninstall: statsmodels
    Found existing installation: statsmodels 0.14.4
    Uninstalling statsmodels-0.14.4:
      Successfully uninstalled statsmodels-0.14.4 0/2 [statsmodels]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pmdarima]ls]
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np

historical = pd.read_csv("MPS_Borough_Level_Crime_Historical.csv")
recent = pd.read_csv("MPS_Borough_Level_Crime_Recent_24_month.csv")

id_cols = ["MajorText", "MinorText", "BoroughName"]
hist_long = historical.melt(id_vars=id_cols, var_name="YearMonth", value_name="CrimeCount")
recent_long = recent.melt(id_vars=id_cols, var_name="YearMonth", value_name="CrimeCount")
crime = pd.concat([hist_long, recent_long], ignore_index=True)

not_boroughs = ["Unknown", "London Heathrow and London City Airports"]
crime = crime[~crime["BoroughName"].isin(not_boroughs)]
crime["Date"] = pd.to_datetime(crime["YearMonth"], format="%Y%m")

london_monthly = crime.groupby("Date")["CrimeCount"].sum().sort_index()
london_monthly = london_monthly.asfreq("MS")

train = london_monthly[:-12]
test = london_monthly[-12:]

print("Train:", len(train), "Test:", len(test))

Train: 179 Test: 12


In [4]:
import pmdarima as pm

auto_model = pm.auto_arima(
    train,
    seasonal=True,
    m=12,                  
    start_p=0, max_p=3,
    start_q=0, max_q=3,
    start_P=0, max_P=2,
    start_Q=0, max_Q=2,
    d=None, D=None,        
    trace=True,            
    error_action="ignore",
    suppress_warnings=True,
    stepwise=True          
)

print(auto_model.summary())

Performing stepwise search to minimize aic
 ARIMA(0,1,0)(0,0,0)[12] intercept   : AIC=3487.949, Time=0.03 sec
 ARIMA(1,1,0)(1,0,0)[12] intercept   : AIC=3475.169, Time=0.05 sec
 ARIMA(0,1,1)(0,0,1)[12] intercept   : AIC=3476.759, Time=0.05 sec
 ARIMA(0,1,0)(0,0,0)[12]             : AIC=3485.954, Time=0.01 sec
 ARIMA(1,1,0)(0,0,0)[12] intercept   : AIC=3484.118, Time=0.03 sec
 ARIMA(1,1,0)(2,0,0)[12] intercept   : AIC=3475.135, Time=0.13 sec
 ARIMA(1,1,0)(2,0,1)[12] intercept   : AIC=3465.122, Time=0.62 sec
 ARIMA(1,1,0)(1,0,1)[12] intercept   : AIC=3464.206, Time=0.12 sec
 ARIMA(1,1,0)(0,0,1)[12] intercept   : AIC=3476.480, Time=0.06 sec
 ARIMA(1,1,0)(1,0,2)[12] intercept   : AIC=3465.122, Time=0.66 sec
 ARIMA(1,1,0)(0,0,2)[12] intercept   : AIC=3476.903, Time=0.13 sec
 ARIMA(1,1,0)(2,0,2)[12] intercept   : AIC=inf, Time=0.90 sec
 ARIMA(0,1,0)(1,0,1)[12] intercept   : AIC=3463.628, Time=0.26 sec
 ARIMA(0,1,0)(0,0,1)[12] intercept   : AIC=3479.476, Time=0.04 sec
 ARIMA(0,1,0)(1,0,0)[12]

In [5]:
forecast_auto = auto_model.predict(n_periods=12)
forecast_auto = pd.Series(forecast_auto, index=test.index)

mae_auto = np.mean(np.abs(test.values - forecast_auto.values))
rmse_auto = np.sqrt(np.mean((test.values - forecast_auto.values)**2))

print("Auto-tuned SARIMA:")
print("MAE:", round(mae_auto, 1))
print("RMSE:", round(rmse_auto, 1))

comparison = pd.DataFrame({
    "Actual": test.values,
    "Auto-tuned Forecast": forecast_auto.values.round(0)
}, index=test.index)
print(comparison)

Auto-tuned SARIMA:
MAE: 3545.2
RMSE: 4039.5
            Actual  Auto-tuned Forecast
Date                                   
2025-03-01   75599              71263.0
2025-04-01   74357              70590.0
2025-05-01   78653              73188.0
2025-06-01   78414              73281.0
2025-07-01   82509              74552.0
2025-08-01   77380              73075.0
2025-09-01   73857              72616.0
2025-10-01   76963              74494.0
2025-11-01   76013              74115.0
2025-12-01   73393              72076.0
2026-01-01   70305              71943.0
2026-02-01   67469              70485.0
